# Decision Tree From Scratch
### Using only Python + Pandas (no sklearn, no AI libs)

**Dataset:** Weather & Play Tennis — predict whether to play tennis based on weather conditions.

**Algorithm:** ID3 (Iterative Dichotomiser 3) — builds the tree by maximizing Information Gain at each split.

---
## Step 1 — Load & Explore the Dataset

In [1]:
import pandas as pd
import math
from collections import Counter

df = pd.read_csv('weather_play.csv')
print(f'Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns')
df

Shape: (14, 5)  →  14 rows, 5 columns


,outlook,temperature,humidity,windy,play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes
5,rainy,cool,normal,True,no
6,overcast,cool,normal,True,yes
7,sunny,mild,high,False,no
8,sunny,cool,normal,False,yes
9,rainy,mild,normal,False,yes


In [2]:
# Class distribution
print('Target distribution:')
print(df['play'].value_counts())
print()
print('Unique values per feature:')
for col in df.columns:
    print(f'  {col:<15}: {df[col].unique().tolist()}')

Target distribution:
play
yes    9
no     5
Name: count, dtype: int64

Unique values per feature:
  outlook        : ['sunny', 'overcast', 'rainy']
  temperature    : ['hot', 'mild', 'cool']
  humidity       : ['high', 'normal']
  windy          : [False, True]
  play           : ['no', 'yes']


---
## Step 2 — Entropy

**Entropy** measures how *mixed* or *impure* a set of labels is.

$$H(S) = -\sum_{c} p_c \cdot \log_2(p_c)$$

| Labels | Entropy |
|---|---|
| All same class | 0.0 (perfectly pure) |
| 50-50 split | 1.0 (maximum impurity) |
| Our dataset (9 yes, 5 no) | ≈ 0.94 |

In [3]:
def entropy(series):
    """Entropy of a pandas Series of labels."""
    counts = series.value_counts()
    probs  = counts / len(series)
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Demonstrate entropy on different distributions
examples = {
    'All YES (pure)'      : pd.Series(['yes'] * 9),
    '50-50 split'         : pd.Series(['yes'] * 7 + ['no'] * 7),
    'Our dataset (9y,5n)' : df['play'],
}

print(f'{"Example":<25} Entropy')
print('-' * 38)
for label, s in examples.items():
    print(f'{label:<25} {entropy(s):.4f}')

Example                   Entropy
--------------------------------------
All YES (pure)            -0.0000
50-50 split               1.0000
Our dataset (9y,5n)       0.9403


---
## Step 3 — Information Gain

**Information Gain** = how much a feature *reduces* entropy after splitting.

$$IG(S, A) = H(S) \;-\; \sum_{v \in A} \frac{|S_v|}{|S|} \cdot H(S_v)$$

We pick the feature with the **highest IG** at each node.

In [4]:
def information_gain(df, feature, target):
    """IG of splitting df on `feature` with respect to `target`."""
    parent_entropy = entropy(df[target])
    n = len(df)

    # Weighted average entropy of each child group
    weighted_child = 0.0
    for value, group in df.groupby(feature):
        weight = len(group) / n
        weighted_child += weight * entropy(group[target])

    return parent_entropy - weighted_child


features = ['outlook', 'temperature', 'humidity', 'windy']

ig_scores = pd.Series(
    {f: information_gain(df, f, 'play') for f in features}
).sort_values(ascending=False)

print('Information Gain per feature:')
print('-' * 35)
for feat, ig in ig_scores.items():
    bar = '█' * int(ig * 40)
    print(f'  {feat:<15} {ig:.4f}  {bar}')

print(f'\n→ Best feature to split on first: [{ig_scores.idxmax()}]')

Information Gain per feature:
-----------------------------------
  outlook         0.2467  █████████
  humidity        0.1518  ██████
  windy           0.0481  █
  temperature     0.0292  █

→ Best feature to split on first: [outlook]


---
## Step 4 — Node Structure

A tree is made of **nodes**:
- **Internal node** — splits on a feature, has child nodes
- **Leaf node** — no more splitting, holds the final prediction

In [5]:
class Node:
    def __init__(self):
        self.feature    = None   # which feature to split on
        self.children   = {}     # {feature_value → child Node}
        self.prediction = None   # only set on leaf nodes
        self.is_leaf    = False

    def __repr__(self):
        if self.is_leaf:
            return f'LeafNode(predict={self.prediction})'
        return f'InternalNode(split_on={self.feature}, branches={list(self.children.keys())})'


# Quick demo
sample_leaf = Node()
sample_leaf.is_leaf = True
sample_leaf.prediction = 'yes'

sample_internal = Node()
sample_internal.feature = 'outlook'
sample_internal.children = {'sunny': sample_leaf, 'rainy': Node()}

print('Leaf node    :', sample_leaf)
print('Internal node:', sample_internal)

Leaf node    : LeafNode(predict=yes)
Internal node: InternalNode(split_on=outlook, branches=['sunny', 'rainy'])


---
## Step 5 — Build the Tree (ID3 Algorithm)

```
build_tree(data, features):
  ┌─ If all labels are the same      → Leaf (return that label)
  ├─ If no features remain           → Leaf (return majority label)
  └─ Otherwise:
       1. Score each feature by Information Gain
       2. Pick the best feature
       3. For each unique value of that feature:
            a. Split the data
            b. Recursively call build_tree on the subset
       4. Return this node with children attached
```

In [6]:
def build_tree(df, features, target, depth=0):
    node = Node()

    # BASE CASE 1: pure node — all labels identical
    if df[target].nunique() == 1:
        node.is_leaf    = True
        node.prediction = df[target].iloc[0]
        return node

    # BASE CASE 2: no features left — predict majority
    if not features:
        node.is_leaf    = True
        node.prediction = df[target].mode()[0]
        return node

    # RECURSIVE CASE: find best feature to split on
    best = max(features, key=lambda f: information_gain(df, f, target))
    node.feature = best
    remaining = [f for f in features if f != best]

    for value, subset in df.groupby(best):
        node.children[value] = build_tree(subset, remaining, target, depth + 1)

    return node


features = ['outlook', 'temperature', 'humidity', 'windy']
tree = build_tree(df, features, 'play')

print('Tree built!')
print('Root node:', tree)
print('Children of root:', {k: str(v) for k,v in tree.children.items()})

Tree built!
Root node: InternalNode(split_on=outlook, branches=['overcast', 'rainy', 'sunny'])
Children of root: {'overcast': 'LeafNode(predict=yes)', 'rainy': 'InternalNode(split_on=windy, branches=[False, True])', 'sunny': "InternalNode(split_on=humidity, branches=['high', 'normal'])"}


---
## Step 6 — Visualize the Tree

In [7]:
def print_tree(node, indent=0, branch='ROOT'):
    pad = '│   ' * indent
    if node.is_leaf:
        print(f'{pad}└─[{branch}] → 🎯 PREDICT: {node.prediction.upper()}')
    else:
        print(f'{pad}└─[{branch}]')
        print(f'{pad}    Split on ⟨{node.feature}⟩')
        for value, child in node.children.items():
            print_tree(child, indent + 1, f'{node.feature} = {value}')

print('DECISION TREE')
print('=' * 55)
print_tree(tree)

DECISION TREE
└─[ROOT]
    Split on ⟨outlook⟩
│   └─[outlook = overcast] → 🎯 PREDICT: YES
│   └─[outlook = rainy]
│       Split on ⟨windy⟩
│   │   └─[windy = False] → 🎯 PREDICT: YES
│   │   └─[windy = True] → 🎯 PREDICT: NO
│   └─[outlook = sunny]
│       Split on ⟨humidity⟩
│   │   └─[humidity = high] → 🎯 PREDICT: NO
│   │   └─[humidity = normal] → 🎯 PREDICT: YES


---
## Step 7 — Predict on New Samples

Walk the tree from root to leaf: at each node, follow the branch matching the sample's feature value.

In [8]:
def predict(node, sample):
    """Predict label for one sample (dict or pd.Series)."""
    if node.is_leaf:
        return node.prediction

    value = sample[node.feature]
    if value not in node.children:
        return 'unknown (unseen value)'

    return predict(node.children[value], sample)


# New test samples
test_df = pd.DataFrame([
    {'outlook': 'sunny',    'temperature': 'hot',  'humidity': 'high',   'windy': 'false'},
    {'outlook': 'overcast', 'temperature': 'cool', 'humidity': 'normal', 'windy': 'true'},
    {'outlook': 'rainy',    'temperature': 'mild', 'humidity': 'normal', 'windy': 'false'},
    {'outlook': 'rainy',    'temperature': 'cool', 'humidity': 'high',   'windy': 'true'},
    {'outlook': 'sunny',    'temperature': 'mild', 'humidity': 'normal', 'windy': 'true'},
])

test_df['prediction'] = test_df.apply(lambda row: predict(tree, row), axis=1)
print('Predictions on new samples:')
test_df

Predictions on new samples:


,outlook,temperature,humidity,windy,prediction
0,sunny,hot,high,false,no
1,overcast,cool,normal,true,yes
2,rainy,mild,normal,false,unknown (unseen value)
3,rainy,cool,high,true,unknown (unseen value)
4,sunny,mild,normal,true,yes


---
## Step 8 — Evaluate on Training Data

In [9]:
df['predicted'] = df.apply(lambda row: predict(tree, row), axis=1)
df['correct']   = df['play'] == df['predicted']

accuracy = df['correct'].mean()
print(f'Training Accuracy: {accuracy*100:.1f}%  ({df["correct"].sum()}/{len(df)} correct)\n')

# Show full result table
result = df[['outlook','temperature','humidity','windy','play','predicted','correct']].copy()
result['correct'] = result['correct'].map({True: '✓', False: '✗'})
result

Training Accuracy: 100.0%  (14/14 correct)



,outlook,temperature,humidity,windy,play,predicted,correct
0,sunny,hot,high,False,no,no,✓
1,sunny,hot,high,True,no,no,✓
2,overcast,hot,high,False,yes,yes,✓
3,rainy,mild,high,False,yes,yes,✓
4,rainy,cool,normal,False,yes,yes,✓
5,rainy,cool,normal,True,no,no,✓
6,overcast,cool,normal,True,yes,yes,✓
7,sunny,mild,high,False,no,no,✓
8,sunny,cool,normal,False,yes,yes,✓
9,rainy,mild,normal,False,yes,yes,✓


---
## Step 9 — Tree Statistics

In [10]:
def tree_stats(node):
    if node.is_leaf:
        return {'depth': 0, 'nodes': 1, 'leaves': 1}
    child_stats = [tree_stats(c) for c in node.children.values()]
    return {
        'depth'  : 1 + max(s['depth']  for s in child_stats),
        'nodes'  : 1 + sum(s['nodes']  for s in child_stats),
        'leaves' : sum(s['leaves'] for s in child_stats),
    }

stats = tree_stats(tree)
print(pd.DataFrame([{
    'Max Depth'     : stats['depth'],
    'Total Nodes'   : stats['nodes'],
    'Leaf Nodes'    : stats['leaves'],
    'Internal Nodes': stats['nodes'] - stats['leaves'],
}]).to_string(index=False))

 Max Depth  Total Nodes  Leaf Nodes  Internal Nodes
         2            8           5               3


---
## Summary

| Concept | What it does |
|---|---|
| **Entropy** | Measures label impurity in a node. Range: 0 (pure) → 1 (mixed) |
| **Information Gain** | How much a feature reduces entropy. Pick the highest at each step |
| **Internal Node** | Asks a question: *what is the value of feature X?* |
| **Leaf Node** | Gives an answer: *predict class Y* |
| **ID3** | The algorithm tying it all together — recursive, greedy |

### What this simple tree does NOT do (yet)
- **Pruning** — the tree memorizes training data; it can overfit
- **Continuous features** — ID3 only handles categorical values natively
- **Train/test split** — all evaluation here is on training data
- **Missing values** — no handling of NaN

These are addressed by more advanced variants: **C4.5**, **CART**, and ensemble methods like **Random Forest** and **Gradient Boosting**.